# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR<sup>2</sup> dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" using the `mlcroissant` library.

### Dataset Source
The dataset source is described via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the FAIR^2 dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

md = dataset.metadata
print(f"Dataset Loaded: {md.name}\nDescription: {md.description}")

## 2. Data Overview
List available record sets and preview their fields by `@id`. All entities are referenced by their `@id` for clarity and reproducibility.

Note: The actual Croissant schema provides structured `recordSet` entries, but to stay robust, we first list all available record sets and explore their field composition.

In [ ]:
# List available record sets by their @id
print("Available Record Sets:")
for rs in dataset.metadata.record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '')}")

### Fields in Each RecordSet
Let's inspect the fields (columns) defined in a chosen record set. We'll display their `@id` and human-readable names. Fields are referenced by their `@id` in column and analysis code blocks.

In [ ]:
# Choose the first (or main) record set to inspect fields
record_sets = list(dataset.metadata.record_sets)
if len(record_sets) == 0:
    raise RuntimeError("No RecordSet found in the provided Croissant schema.")
main_rs = record_sets[0]['@id']
main_rs_def = None
for rs in dataset.metadata.record_sets:
    if rs['@id'] == main_rs:
        main_rs_def = rs
        break
print(f"Fields (columns) in record set {main_rs}:")
for field in main_rs_def['field']:
    fname = field.get('name', field['@id'])
    print(f"  - {field['@id']}: {fname}  [{field.get('dataType')}] ")

## 3. Data Extraction
Load data from all available record sets. Each record set and its fields are referenced by their `@id` for auditing purposes.
We'll demonstrate loading all records for the main record set into a DataFrame for further analysis.

In [ ]:
# Extract data from all record sets and load to dataframes
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dfs = {}

for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    # Records as list of dictionaries (fields referenced by @id)
    records = list(dataset.records(record_set=rs_id))
    dfs[rs_id] = pd.DataFrame(records)
    print(f"  ...{len(records)} records loaded.")

# Display column names of the main record set
print(f"\nColumns available in main record set ({main_rs}):")
print(dfs[main_rs].columns.tolist())

# Preview data
dfs[main_rs].head()

## 4. Exploratory Data Analysis (EDA)
Next, we'll select a numeric field (by `@id`) from the main record set and perform some basic EDA: filtering and normalization, and group-wise statistics.  
**All references use the field's `@id` exactly as it appears in the schema.**

In [ ]:
# Find a numeric field (@id) from the main record set
numeric_field = None
group_field = None
for field in main_rs_def['field']:
    if field.get('dataType') in ['Float', 'Integer', 'Number'] and numeric_field is None:
        numeric_field = field['@id']
    if (field.get('dataType') in ['Text', 'String']) and group_field is None:
        group_field = field['@id']
if numeric_field is None:
    raise RuntimeError("No numeric field found in record set.")
print(f"Numeric field used for analysis: {numeric_field}")
if group_field is None:
    print("No grouped field found: skipping grouped analysis.")
else:
    print(f"Grouping field for analysis: {group_field}")

# Perform basic filtering, normalization, and grouping
df = dfs[main_rs]
# Drop records with missing values in selected fields
filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce').notnull()].copy()
filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field])
# Filter: keep records where value > some threshold (e.g., 10)
threshold = 10
gt_mask = filtered_df[numeric_field] > threshold
filtered_df = filtered_df.loc[gt_mask]
print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
print(filtered_df[[numeric_field]].head())

# Normalize numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a text field if available
if group_field is not None and group_field in filtered_df:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
    print(f"\nGrouped data by field: {group_field} (showing mean of {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the chosen numeric field and the effect of normalization. We'll use a histogram for the numeric field and a box plot grouped by the group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.histplot(filtered_df[numeric_field], kde=True, bins=30, color='tab:blue')
plt.title(f"Histogram of {numeric_field}")
plt.xlabel(numeric_field)

plt.subplot(1, 2, 2)
sns.histplot(filtered_df[f'{numeric_field}_normalized'], kde=True, bins=30, color='tab:green')
plt.title(f"Normalized {numeric_field}")
plt.xlabel(f'{numeric_field} (z-score)')

plt.tight_layout()
plt.show()

# Boxplot by group if possible
if group_field is not None and group_field in filtered_df:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=60, ha='right')
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, analyze, and visualize mass spectrometry protein analysis data using the `mlcroissant` library.

By referencing all Croissant entities (record sets, fields, columns) by their `@id`, you ensure fully FAIR and reproducible data access. This approach is robust to schema evolution and best practice for interoperable research.

For advanced analyses, you can select different fields and aggregate, filter, or visualize as needed, referencing the schema's `@id` at every step.